# Text Classification using Naive Bayes (Very Simple Step-by-Step)

This notebook is written in a very simple style.

How to use it:
- Run one cell at a time from top to bottom.
- Read every output before going to the next cell.
- If one step fails, fix that step first.

Goal: predict which category a text belongs to using Naive Bayes.

## Step 0: Import Libraries

What is happening here:
- We import the tools needed for data loading, text processing, plotting, and modeling.

Why this step matters:
- Without these libraries, the notebook cannot run.
- Each library has a clear role in the lesson.

In [ ]:
# Cell 3: Optional install command if packages are missing.
# Run this only if the next import cell fails.
# %pip install numpy pandas scikit-learn matplotlib seaborn

In [1]:
# Cell 4: Import libraries.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

SEED = 42
print('All libraries loaded successfully.')

All libraries loaded successfully.


## Step 1: Find and Load the Dataset

What is happening here:
- We search common folders for the CSV file.
- We load it into a pandas DataFrame.

Why this step matters:
- We need the dataset before we can train any model.
- Keeping this step simple makes the notebook easier to reuse.

In [2]:
# Cell 6: Locate the dataset and load it.
possible_paths = [
    Path.cwd() / 'data' / 'raw' / 'synthetic_text_data.csv',
    Path.cwd().parent / 'data' / 'raw' / 'synthetic_text_data.csv',
    Path.cwd() / 'synthetic_text_data.csv',
    Path.cwd().parent / 'synthetic_text_data.csv',
]

for path in possible_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('Could not find synthetic_text_data.csv in the project root or data/raw/.')

data = pd.read_csv(csv_path)
print(f'Loaded file: {csv_path}')
print(f'Shape: {data.shape}')
data.head()

Loaded file: /Users/bti-001541/machine-learning-project/data/raw/synthetic_text_data.csv
Shape: (80, 2)


,text,label
0,The football team scored a winning goal in the...,Sports
1,The striker celebrated after scoring three goa...,Sports
2,The coach designed a strong training plan for ...,Sports
3,Fans cheered loudly in the stadium during the ...,Sports
4,The basketball player practiced shooting befor...,Sports


## Step 2: Split Text and Labels

What is happening here:
- We store the text column in `X`.
- We store the label column in `y`.

Why this step matters:
- In machine learning, inputs and outputs are usually handled separately.
- This keeps the next steps simple.

In [3]:
# Cell 8: Split text and labels.
X = data['text']
y = data['label']
print('Number of text rows:', len(X))
print('Unique labels:', sorted(y.unique()))
pd.DataFrame({'text': X.head(), 'label': y.head()})

Number of text rows: 80
Unique labels: ['Entertainment', 'Politics', 'Sports', 'Technology']


,text,label
0,The football team scored a winning goal in the...,Sports
1,The striker celebrated after scoring three goa...,Sports
2,The coach designed a strong training plan for ...,Sports
3,Fans cheered loudly in the stadium during the ...,Sports
4,The basketball player practiced shooting befor...,Sports


## Step 3: Split the Data into Training and Testing Sets

What is happening here:
- We keep most rows for training.
- We keep some rows for testing.

Why this step matters:
- The model learns on training data.
- The test data helps us check if it works on unseen text.

In [4]:
# Cell 10: Split the dataset into train and test parts.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

Training rows: 64
Testing rows: 16


## Step 4: Convert Text into Numeric Features

What is happening here:
- We use `CountVectorizer` to convert words into counts.
- This turns text into numbers.

Why this step matters:
- Machine learning models cannot read raw text directly.
- Word counts are one of the simplest and easiest text features.

In [5]:
# Cell 12: Convert text to word-count features.
vectorizer = CountVectorizer(ngram_range=(1, 2))
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print('Training matrix shape:', X_train_vectorized.shape)
print('Testing matrix shape:', X_test_vectorized.shape)

Training matrix shape: (64, 741)
Testing matrix shape: (16, 741)


**Inference:**
- Each row is now a text example turned into numbers.
- Each column represents a word or a small word pair learned from the training data.
- This is how raw text becomes something the model can understand.

## Step 5: Train the Naive Bayes Classifier

What is happening here:
- We train a `MultinomialNB` model using the training data.

Why this step matters:
- This is the learning step.
- The model studies word patterns for each category.

In [6]:
# Cell 15: Train the Naive Bayes model.
model = MultinomialNB()
model.fit(X_train_vectorized, y_train)
print('Model training completed.')

Model training completed.


**Inference:**
- The model has now learned which words are common in each category.
- For example, words like `goal` may suggest Sports, and words like `software` may suggest Technology.

## Step 6: Make Predictions

What is happening here:
- We ask the trained model to predict labels for the test data.

Why this step matters:
- This lets us compare the model's guesses with the real answers.

In [7]:
# Cell 18: Predict the test labels.
y_pred = model.predict(X_test_vectorized)
prediction_table = pd.DataFrame({
    'text': X_test.reset_index(drop=True),
    'actual_label': y_test.reset_index(drop=True),
    'predicted_label': pd.Series(y_pred),
})
prediction_table.head()

,text,actual_label,predicted_label
0,The cloud backup system protects important files,Technology,Sports
1,The programmer wrote code for a new web applic...,Technology,Entertainment
2,The audience enjoyed the magic performance,Entertainment,Entertainment
3,The opposition criticized the new decision str...,Politics,Technology
4,The dancer performed beautifully on stage,Entertainment,Entertainment


**Inference:**
- This table shows what the model guessed for each test sentence.
- If actual and predicted labels match, that prediction is correct.

## Step 7: Evaluate the Model

What is happening here:
- We calculate accuracy.
- We build a confusion matrix.
- We print a classification report.

Why this step matters:
- Evaluation tells us how well the model performed.
- It also helps us see where the mistakes happened.

In [ ]:
# Cell 21: Evaluate the model.
accuracy = accuracy_score(y_test, y_pred)
class_labels = np.unique(y_test)
conf_matrix = confusion_matrix(y_test, y_pred, labels=class_labels)
report_text = classification_report(y_test, y_pred, zero_division=0)

print(f'Accuracy: {accuracy * 100:.2f}%')
print('Classification Report:')
print(report_text);

SyntaxError: unterminated string literal (detected at line 9) (1918409356.py, line 9)

**Inference:**
- Accuracy tells us what percentage of test texts were classified correctly.
- A higher accuracy means the model made more correct predictions.
- The classification report gives more detail for each category.

## Step 8: Visualize the Confusion Matrix

What is happening here:
- We draw the confusion matrix as a heatmap.

Why this step matters:
- It shows which categories are predicted correctly and which ones get mixed up.

In [ ]:
# Cell 24: Plot the confusion matrix.
plt.figure(figsize=(8, 6))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels,
)
plt.title('Confusion Matrix Heatmap')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

**Inference:**
- The diagonal cells show correct predictions.
- The off-diagonal cells show mistakes.
- Darker diagonal values usually mean the model is performing well for those categories.

## Step 9: Predict on Unseen Text

What is happening here:
- We give the model a new sentence it has not seen before.
- Then we check which category it predicts.

Why this step matters:
- This shows how the trained model can be used in a real situation.

In [ ]:
# Cell 27: Predict one new sentence.
user_input = 'I love artificial intelligence and machine learning'
user_input_vectorized = vectorizer.transform([user_input])
predicted_label = model.predict(user_input_vectorized)
print(f"The input text belongs to the '{predicted_label[0]}' category.")

**Inference:**
- The model uses the words in the new sentence to guess the category.
- If the sentence contains words like `artificial intelligence` and `machine learning`, Technology is a reasonable prediction.

## Summary: Complete Naive Bayes Flow (From Start to Finish)

### What Did We Build?
We built a simple text classification model that learns to place short text documents into categories like Sports, Technology, Politics, and Entertainment.

---

### FLOW MATRIX: How the Model Works

#### Stage 1: Load the Data
```
CSV file
↓
Load text and labels into a DataFrame
```

#### Stage 2: Split Text and Labels
```
Text column → X
Label column → y
```

#### Stage 3: Train/Test Split
```
Training data → learn patterns
Testing data → check performance
```

#### Stage 4: Convert Text to Numbers
```
CountVectorizer
↓
Words become count-based numeric features
```

#### Stage 5: Train Naive Bayes
```
Model studies word patterns in each class
↓
Learns which words suggest which category
```

#### Stage 6: Predict
```
New text enters the model
↓
Model picks the most likely category
```

#### Stage 7: Evaluate
```
Accuracy + classification report + confusion matrix
↓
Understand model quality and mistakes
```

#### Stage 8: Use on Unseen Text
```
Give one new sentence
↓
Get the predicted category
```

---

### Simple Child Analogy
Imagine sorting cards into boxes:
1. You read the words on the card.
2. You remember which words usually belong to which box.
3. When you see a new card, you put it in the box that looks most similar.
That is what Naive Bayes does with text.

---

### Final Learning Summary
You just built a complete beginner-friendly text classification pipeline that:
1. Loads a dataset
2. Separates text and labels
3. Splits train and test data
4. Converts words into numbers
5. Trains a Naive Bayes model
6. Evaluates the model
7. Visualizes mistakes
8. Predicts a new unseen sentence